<p style="text-align: center">
<img src="../../../assets/images/dtlogo.png" alt="Duckietown" width="50%">
</p>

# 🚙 02 - Wheel Calibration

<!--
💻 🚙
-->

In this activity we will calibrate the motor / wheel assemblies of our robots by determining two calibration parameters, to ensure that:

1. Duckiebots go (reasonably) straight when commanded to do so, and
2. the wheels do not slip. 

You will need the Duckietown hardware to complete this activity. 

If you do not have a Duckiebot and Duckietown, proceed to the next activity: the [wheel encoders tutorial](../03-Wheel-Encoders-Tutorial/wheel_encoders_tutorial.ipynb).

## Motivation

We have derived a model of the Duckiebot's motors and kinematics:

<p style="text-align:center;">
<img src="../../../assets/images/wheel-calibration/model.png" alt="Drawing" style="width: 500px;"/>
</p>

To obtain it, we introduced simplifying assumptions: 

1. The robot is symmetric along the the longitudinal axis;
2. The wheels do not slip.

Unfortunately, neither of these two assumptions really holds in practice. As a result: 

1. The robot will not drive straight when receiving equal commands to the wheels;
2. The wheels (might) slip. 

To mitigate these effects, we would like to:

1. Modify slightly the input to one or the other motor, to make sure the output the same travelled distance when provided with the same input;
2. Easily modify the overall Duckiebot speed, which is one of the most significant causes of slip (the slower the velocity, the lower the slipping).

In this activity we will calibrate the wheels, i.e., introduce and find two parameters that allow us to mitigate these undesired effects.

## Approach

Let's look at the relationship we derived between the voltage provided to each motors and the resulting angular velocity of the wheel:

$$ V_t = \left( R\frac{b}{K_i} + K_b \right) \dot \phi_t $$

The coefficient is a function of the wheel radius and the motor constants, which are difficult to estimate. We therefore lump all terms in a single unknown motor coefficient ($m$). Writing the same for right and left wheels, we get two motor coefficients to estimate: $m_r$ and $m_l$. 

To provide some physical insight, we reparametrize these two unknowns in two physically meaningful quantities, the _gain_ ($g$) and _trim_ ($t$), so that the relation can be thought as:

$$ V_{r,t} = \left( g + t \right) \dot \phi_{r,t} $$
$$ V_{l,t} = \left( g - t \right) \dot \phi_{l,t} $$

In this way we always have two parameters to find, but we know that increasing (decreasing) the gain will make the robot go faster (slower), while adjusting the trim value will correct asymmetries in the driving. (Note that in the physical implementation everything will be scaled by a constant - but no need to worry about that for now). 

## Calibrating the gain ($g$)

The gain parameter can have any value betwen 0 and 1. The default is set as 1. 

The objective of this first section is to determine a value of the gain that works well for your Duckiebot. There isn't a right or wrong number, the best value will vary depending on the desired driving experience. 

As a rule of thumb, we will want to set the gain for our robot to _the highest value that does not make the wheels slip_. 

Note that slippage will depend on the surface you drive your Duckiebot on, so we suggest to calibrate the gain while driving on the Duckietown tiles. 

### Gain calibration procedure

1. Place your powered on Duckiebot on your Duckietown.

2. Open a terminal, navigate to the **repository root** (`duckietown-rewritten/`), and deploy the ModCon task to your robot:

       python launch.py --run --bot ROBOTNAME --task modcon

   Replace `ROBOTNAME` with your Duckiebot's hostname.

3. Open the URL printed by the command in your browser (or navigate directly to `http://ROBOTNAME.local:5000`).

4. In the web interface, find the **Calibration** panel. The default gain is `1.0`.

5. Use the **Test Drive** button to drive your robot forward for a short distance. Observe whether the wheels slip. If they do, reduce the gain value and try again.

6. Adjust the gain until you find the highest value that does not cause wheel slip on the Duckietown tiles.

#### Save the new gain

Click **Save Calibration** in the web interface to persist your gain value to `config/modcon_config.yaml`. This file is used every time the task runs.

#### Final check

You can verify your saved value by opening `config/modcon_config.yaml` in the repository and checking the `gain` field.

## Calibrating the trim ($t$)

This procedure is similar to that of calibrating the gain.

### Trim calibration procedure

1. Finish calibrating the gain by following the steps above before proceeding further.

2. Place your Duckiebot on a lane, aligned with the road:

<p style="text-align: center">
<img src="../../../assets/images/wheel-calibration/wheel-calib-setup.jpg" alt="Drawing" style="width: 300px;"/>
</p>

3. In the web interface (running at `http://ROBOTNAME.local:5000`), drive straight for roughly 2 meters (3 tiles).

4. Measure how far the robot ended from the yellow line:
   - If the robot drifted **left**, increase the trim (positive values steer more towards left)
   - If the robot drifted **right**, decrease the trim (negative values steer more towards right)

5. Adjust the trim value in the **Calibration** panel and repeat until the robot drives reasonably straight.

Note: if you remove and reattach the wheels at any point, you may need to re-calibrate the trim.

### Saving the new trim value

Click **Save Calibration** in the web interface to persist your trim value to `config/modcon_config.yaml`.

#### Final check

Open `config/modcon_config.yaml` and verify that the `trim` field reflects your chosen value.

## Other kinematic parameters

You might have noticed that `trim` and `gain` are not the only kinematic parameters. There are other constants and constraints that govern how the Duckiebot drives safely.

All parameters are stored in `config/modcon_config.yaml`. Here is a brief description of each:

- **gain** (`float`): scaling factor applied to the desired velocity. Default: `1.0`

- **trim** (`float`): corrects asymmetries between left and right motors so the robot drives straight. Default: `0.0`

- **baseline** (`float`): distance between the centers of the two wheels (meters). Default: `0.1`

- **radius** (`float`): radius of the wheel (meters). Default: `0.0318`

- **k** (`float`): motor constant, assumed equal for both motors. Default: `27.0`

- **power_limit** (`float`): limits the final PWM commands sent to the motors. Default: `0.85`

- **v_max** (`float`): limits the maximum input linear velocity (m/s). Default: `1.0`

- **omega_max** (`float`): limits the maximum input angular velocity (rad/s). Default: `8.0`

We do not recommend modifying the other parameters unless you have a specific reason. If your Duckiebot rotates too quickly, you can reduce `omega_max` in `config/modcon_config.yaml` (e.g., `omega_max: 5.0` provides a joyful driving experience).

You can now move on to the [wheel encoder tutorial](../03-Wheel-Encoders-Tutorial/wheel_encoders_tutorial.ipynb) activity. 